# Intro

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

input_path_str: str = os.getenv("FEEDBACK_SOURCE_DIR_PATH") or '.'


In [ ]:
# from nltk.sentiment import SentimentIntensityAnalyzer
# import nltk 

# # nltk.download('vader_lexicon')
# sia = SentimentIntensityAnalyzer()

# def nullable_polarity_scores(text):
#     if pd.isna(text):
#         return {"neg": np.nan, "neu": np.nan, "pos": np.nan, "compound": np.nan}
#     return sia.polarity_scores(str(text))

In [ ]:
from enum import StrEnum, auto
from pathlib import Path
import pandas as pd

class DataFileType(StrEnum):
    CLEAN = auto()
    ENHANCED = auto()

class DataFilePeriod(StrEnum):
    FIRST_MONTH = '20d'
    HALF_YEAR = '6m'
    FULL_YEAR = '1y'

class InputManager:
    _type: DataFileType
    _period: DataFilePeriod
    _SOURCE_DIR_PATH: Path = Path(input_path_str)
    _SUFFIX: str = '.csv'

    def _create_input_path(self, file_type: DataFileType, data_period: DataFilePeriod) -> Path:
        return (self._SOURCE_DIR_PATH / (file_type + data_period).upper()).with_suffix(self._SUFFIX)

    def get_input_data(self, file_type: DataFileType, data_period: DataFilePeriod) -> pd.DataFrame:
        path: Path = self._create_input_path(file_type, data_period)
        df = pd.read_csv(path, index_col=0)
        return df

inputs: InputManager = InputManager()
preferred_file_type: DataFileType = DataFileType.ENHANCED

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List

@dataclass
class PreprocessingStrategy:
    type_map: Dict[str, Any] = field(default_factory=dict)              # simple dtypes
    datetime_cols: List[str] = field(default_factory=list)
    categorical_cols: List[str] = field(default_factory=list)
    ordinal_cols: Dict[str, List[Any]] = field(default_factory=dict)    # Map column name to an ordered list of categories, lowest to highest

@dataclass
class DataPreprocessor:
    _df: pd.DataFrame

    def set_dtypes(self, strategy: PreprocessingStrategy) -> None:
        df: pd.DataFrame = self._df.copy()

        for col in strategy.datetime_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col])

        for col in strategy.categorical_cols:
            if col in df.columns:
                df[col] = df[col].astype("category")

        for col, categories in strategy.ordinal_cols.items():
            if col in df.columns:
                # Enforce exact order and flag as ordered
                ordinal_type = pd.CategoricalDtype(categories=categories, ordered=True)
                df[col] = df[col].astype(ordinal_type)

        valid_type_map = {col: dtype for col, dtype in strategy.type_map.items() if col in df.columns}
        df = df.astype(valid_type_map)

        self._df = df

# Survey 20 Days with SOP

In [ ]:
current_data_period: DataFilePeriod = DataFilePeriod.FIRST_MONTH

df = inputs.get_input_data(preferred_file_type, current_data_period)
df['department'] = df['department'].fillna('Unspecified')
df.insert(0, 'posting_timestamp', df['posting_day'] + ' ' + df['posting_hour'])
df.drop(['posting_day', 'posting_hour'], axis=1, inplace=True)

strategy: PreprocessingStrategy = PreprocessingStrategy(
    type_map={
        'positive_feedback': 'string[pyarrow]', 
        'negative_feedback': 'string[pyarrow]', 
        'positive_feedback_EN': 'string[pyarrow]', 
        'negative_feedback_EN': 'string[pyarrow]',
        'positive_feedback_EN_neg': 'float32',
        'positive_feedback_EN_neu': 'float32',
        'positive_feedback_EN_pos': 'float32',
        'positive_feedback_EN_compound': 'float32',
        'negative_feedback_EN_neg': 'float32',
        'negative_feedback_EN_neu': 'float32',
        'negative_feedback_EN_pos': 'float32',
        'negative_feedback_EN_compound': 'float32',
    },
    datetime_cols=['posting_timestamp'],
    categorical_cols=['survey_type', 'department'],
    ordinal_cols={'onboarding_rating': list(range(1,6))}
)

preprocessor: DataPreprocessor = DataPreprocessor(df)
preprocessor.set_dtypes(strategy)
df = preprocessor._df
df

## Univariate Analysis

In [ ]:
print(df.shape)
print('---\n')
print(df.info())
print('---\n')
print(df.head())
print('---\n')
print('The indicies are unique: ', df.index.nunique() == len(df))

### Datetime variables

Datetime variables require specialized analysis because time functions both as an index (ordering events sequentially) and a feature (reflecting cyclical human behavior and seasonality).
The main goal of datetime univariate analysis is to evaluate data continuity, spot missing time gaps, assess time resolution, and understand periodic patterns.

#### posting_timestamp

In [ ]:
current_series = df['posting_timestamp']
t_min, t_max = current_series.min(), current_series.max()
total_span = (t_max - t_min)
time_deltas = current_series.diff()

def td_format(td_object):
    seconds = int(td_object.total_seconds())
    periods = [
        ('year',        60*60*24*365),
        ('month',       60*60*24*30),
        ('day',         60*60*24),
        ('hour',        60*60),
        ('minute',      60),
        ('second',      1)
    ]

    strings=[]
    for period_name, period_seconds in periods:
        if seconds > period_seconds:
            period_value , seconds = divmod(seconds, period_seconds)
            has_s = 's' if period_value > 1 else ''
            strings.append("%s %s%s" % (period_value, period_name, has_s))

    return " ".join(strings)

print(f'''
Temporal coverage
    t_min: {t_min:%Y/%m/%d}
    t_max: {t_max:%Y/%m/%d}
    total_span: {td_format(total_span)}
---
Sampling Interval & Resolution
    median difference between answer times: {td_format(time_deltas.median())}
    average difference between answer times: {td_format(time_deltas.mean())}
''')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_datetime = df.dropna(subset=['posting_timestamp']).copy()

df_datetime['Year-Month'] = [ts.strftime('%Y-%m') for ts in df_datetime['posting_timestamp']]
df_datetime['DayOfWeek']  = [ts.strftime('%a') for ts in df_datetime['posting_timestamp']]
df_datetime['Hour']       = [ts.hour for ts in df_datetime['posting_timestamp']]

day_order = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

monthly_map = {}
for ym in df_datetime['Year-Month']:
    monthly_map[ym] = monthly_map.get(ym, 0) + 1

if monthly_map:
    min_ym = min(monthly_map.keys())
    max_ym = max(monthly_map.keys())
    
    start_y, start_m = map(int, min_ym.split('-'))
    end_y, end_m     = map(int, max_ym.split('-'))
    
    all_months = []
    curr_y, curr_m = start_y, start_m
    while (curr_y, curr_m) <= (end_y, end_m):
        all_months.append(f"{curr_y:04d}-{curr_m:02d}")
        curr_m += 1
        if curr_m > 12:
            curr_m = 1
            curr_y += 1
            
    monthly_keys = all_months
    monthly_vals = [monthly_map.get(ym, 0) for ym in monthly_keys]
else:
    monthly_keys = []
    monthly_vals = []

daily_map = {}
for d in df_datetime['DayOfWeek']:
    daily_map[d] = daily_map.get(d, 0) + 1

daily_vals = [daily_map.get(d, 0) for d in day_order]

hourly_map = {}
for h in df_datetime['Hour']:
    hourly_map[h] = hourly_map.get(h, 0) + 1

hourly_vals = [hourly_map.get(h, 0) for h in range(24)]

y_max = max(max(monthly_vals, default=0), max(daily_vals, default=0), max(hourly_vals, default=0)) * 1.15

fig, axes = plt.subplots(
    1, 3, 
    figsize=(18, 5), 
    sharey=True, 
    gridspec_kw={'width_ratios': [1.8, 0.6, 1.6]}
)

sns.barplot(
    x=monthly_keys,
    y=monthly_vals,
    hue=monthly_keys,
    legend=False,
    ax=axes[0],
    palette='Blues_d'
)
axes[0].set_title('Post Frequency by Month')
axes[0].set_ylabel('Number of Posts')
axes[0].set_xlabel('Year-Month')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(
    x=day_order, 
    y=daily_vals, 
    hue=day_order, 
    legend=False, 
    ax=axes[1], 
    palette='Blues_d'
)
axes[1].set_title('Post Frequency by Day')
axes[1].set_xlabel('DayOfWeek')
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(
    x=list(range(24)), 
    y=hourly_vals, 
    hue=list(range(24)), 
    legend=False, 
    ax=axes[2], 
    palette='viridis'
)
axes[2].set_title('Post Frequency by Hour of Day')
axes[2].set_xlabel('Hour')
axes[2].set_xticks([i - 0.5 for i in range(25)])
axes[2].set_xticklabels([f"{h:02d}:00" for h in range(25)], rotation=45)

for ax in axes:
    for container in ax.containers:
        labels = [f'{int(v)}' if v > 0 else '' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=2, fontsize=8)

axes[0].set_ylim(0, y_max)

plt.tight_layout()
plt.show()

Note: the y-axis is common for all 3 plots. Additionally, given the intuition of something happening IN a given month or ON a given day of the week, the tickmarks under the first 2 plots are directly under the bars. However, the same cannot be said about hours, therefore, the tickmarks were explicitly shifted to highlight that the bar in between, say, 14:00 and 15:00 ticks show counts for posts in time range [14:00, 15:00).

Note: by "posting" here we mean the event of a feedback message being visible to the members of channel #15-feedback

In [ ]:
df_datetime = df.dropna(subset=['posting_timestamp']).copy()

df_datetime['DayOfWeek'] = df_datetime['posting_timestamp'].dt.day_name()
labels = ['00:00-03:59', '04:00-07:59', '08:00-11:59', '12:00-15:59', '16:00-19:59', '20:00-23:59']

df_datetime['Hour_4H'] = pd.cut(
    df_datetime['posting_timestamp'].dt.hour, 
    bins=[-1, 3, 7, 11, 15, 19, 23], 
    labels=labels
)

day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

heatmap_data = pd.crosstab(
    df_datetime['DayOfWeek'], 
    df_datetime['Hour_4H']
).reindex(day_order)

plt.figure(figsize=(12, 5))
sns.heatmap(heatmap_data, cmap='YlGnBu', annot=True, cbar_kws={'label': 'Post Count'})
plt.title('Post Frequency Heatmap (Day of Week vs. Hour of Day)')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

The start of the week has been set to Sunday to highlight the pattern in the data.

### Nominal variables

Nominal variables are such categorical variables where categories have no inherent order or ranking.
The univariate analysis here focuses on frequency distributions, central tendency (mode), and cardinality/entropy.

#### survey_type

Just sanity checks, this variable here is not necessary from the point of view of the decision making. We expect it to be not null and constant everywhere; `survey_type='20 Days'`.

In [ ]:
st = df['survey_type']
is_valid = (st.eq('20 Days')).all()
print(f'''
The variable survey_type is valid: {is_valid}
''')

#### department

Department the reposndent declared they belong to.

O - operacyjny; 
B - biznesowy; 
P - projektowy; 
T - techniczny

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

current_series = df['department']

total_obs = len(current_series)
freq_map = {}
for item in current_series:
    # key = '<NA>' if item is None or (hasattr(item, 'isna') and item.isna()) or str(item) in ('nan', '<NA>') else str(item)
    key = str(item)
    freq_map[key] = freq_map.get(key, 0) + 1

# na_count = freq_map.pop('<NA>', 0)
na_count = freq_map.pop('Unspecified', 0)

sorted_valid = sorted(freq_map.items(), key=lambda x: x[1], reverse=True)
categories = [k for k, _ in sorted_valid] + ['Unspecified']
counts = [v for _, v in sorted_valid] + [na_count]

fig, ax = plt.subplots(figsize=(7, 4.5))

# palette = ['gray' if cat == '<NA>' else 'tab:blue' for cat in categories]
palette = ['gray' if cat == 'Unspecified' else 'tab:blue' for cat in categories]

sns.barplot(
    x=categories,
    y=counts,
    hue=categories,
    legend=False,
    ax=ax,
    palette=palette
)

ax.set_title('Department Frequency Distribution')
ax.set_xlabel('Department')
ax.set_ylabel('Count')

for container in ax.containers:
    labels = [f'{int(v)}\n({v/total_obs:.2%})' for v in container.datavalues]
    ax.bar_label(container, labels=labels, padding=3, fontsize=8)

ax.set_ylim(0, max(counts) * 1.15)

plt.tight_layout()
plt.show()

Note: department='B' has a huge dip in the reposonse rate, it won't be the best idea to include it in the bivariate analysis just now. 

In [ ]:
mode_key, mode_val = sorted_valid[0]
antimode_key, antimode_val = sorted_valid[-1]

print(f'''
The mode category is "{mode_key}" with value {mode_val}, constituting therefore {mode_val/total_obs:.2%} of all observations.
The antimode category is "{antimode_key}" with value {antimode_val}, constituting therefore {antimode_val/total_obs:.2%} of all observations.
There are {na_count} answer{'s' if na_count != 1 else ''} with no department specified, constituting therefore {na_count/total_obs:.2%} of all observations.
''')

### Ordinal variables

#### onboarding_rating

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

current_series = df['onboarding_rating']
total_obs = len(current_series)
ordinal_order = list(current_series.dtype.categories)

freq_map = {}
for item in current_series:
    key = 'Unspecified' if item is None or (hasattr(item, 'isna') and item.isna()) or str(item) in ('nan', 'Unspecified') else str(item)
    freq_map[key] = freq_map.get(key, 0) + 1

categories = [str(cat) for cat in ordinal_order] + ['Unspecified']
counts = [freq_map.get(cat, 0) for cat in categories]

fig, ax = plt.subplots(figsize=(8, 4.5))

palette = ['gray' if cat == 'Unspecified' else 'tab:blue' for cat in categories]

sns.barplot(
    x=categories,
    y=counts,
    hue=categories,
    legend=False,
    ax=ax,
    palette=palette
)

ax.set_title('Onboarding Rating Frequency Distribution')
ax.set_xlabel('Onboarding Rating')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)

for container in ax.containers:
    labels = [f'{int(v)}\n({v/total_obs:.2%})' for v in container.datavalues]
    ax.bar_label(container, labels=labels, padding=3, fontsize=8)

ax.set_ylim(0, max(counts, default=1) * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
*ordered_valid, na_counter = list(zip(categories, counts))
na_count = na_counter[1]
average_score = sum(int(cat) * count for (cat, count) in ordered_valid)/total_obs
median_score = current_series.cat.codes[current_series.cat.codes >= 0].median()
mode_key, mode_val = max(ordered_valid, key=lambda x: x[1])
antimode_key, antimode_val = min(ordered_valid, key=lambda x: x[1])


print(f'''
The average score is "{average_score:.2f}".
The median score is "{int(median_score) if int(median_score) == median_score else median_score}".
The mode score is "{mode_key}" with value {mode_val}, constituting therefore {mode_val/total_obs:.2%} of all observations.
The antimode score is "{antimode_key}" with value {antimode_val}, constituting therefore {antimode_val/total_obs:.2%} of all observations.
There are {na_count} answer{'s' if na_count != 1 else ''} with no score specified, constituting therefore {na_count/total_obs:.2%} of all observations.
''')

average_score

### Free text variables

Univariate analysis of free-text (unstructured text) data focuses on summarizing the distribution, statistical properties, length characteristics, and dominant vocabulary or themes of a single text column.

#### positive_feedback

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['positive_feedback']
lang = 'PL'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', '<NA>'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

Lexical Richness (or Lexical Diversity) evaluates the variety and breadth of vocabulary used in a text corpus relative to its total length. Here measured by Corpus and Mean Document-Level TTR.

Type-Token Ratio (TTR):
$$\text{TTR} = \frac{\text{Unique Words (Types)}}{\text{Total Words (Tokens)}}$$

Scale: $0.0$ to $1.0$.

Interpretation: Higher values indicate richer vocabulary. 

However, TTR naturally drops as text length increases because common words repeat.


In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.8, not 0.0.

Note: CTTR < MDLTTR => Feedback is consise (volunteers don't write much) and focused (repeated topics).

#### negative_feedback

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['negative_feedback']
lang = 'PL'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', 'Unspecified'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.8, not 0.0.

Note: CTTR < MDLTTR => Feedback is consise (volunteers don't write much) and focused (repeated topics).

#### positive_feedback_EN

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['positive_feedback_EN']
lang = 'EN'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', '<NA>'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.7, not 0.0.

Note: CTTR < MDLTTR => Feedback is consise (volunteers don't write much) and focused (repeated topics).

#### negative_feedback_EN

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['negative_feedback_EN']
lang = 'EN'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', '<NA>'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.7, not 0.0.

Note: CTTR < MDLTTR => Feedback is consise (volunteers don't write much) and focused (repeated topics).

### Continuous variables

When performing univariate analysis on continuous variables, the focus shifts to distribution shape, central tendency, spread, tail behavior, and data integrity.

#### Sentiment variables: [positive|negative]\_feedback_EN\_[neg|neu|pos|compound]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

grid_layout = [
    ['positive_feedback_EN_neg', 'positive_feedback_EN_neu', 'positive_feedback_EN_pos', 'positive_feedback_EN_compound'],
    ['negative_feedback_EN_neg', 'negative_feedback_EN_neu', 'negative_feedback_EN_pos', 'negative_feedback_EN_compound']
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True, sharex='col')

for row_idx in range(2):
    for col_idx in range(4):
        ax = axes[row_idx, col_idx]
        col_name = grid_layout[row_idx][col_idx]
        current_series = df[col_name].dropna()
        
        mean_val = current_series.mean() if not current_series.empty else 0
        median_val = current_series.median() if not current_series.empty else 0
        
        sns.histplot(current_series, kde=True, ax=ax, color='tab:blue', bins=30)
        
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.2, label=f'Mean: {mean_val:.2f}')
        ax.axvline(median_val, color='green', linestyle='-', linewidth=1.2, label=f'Med: {median_val:.2f}')
        
        ax.set_title(col_name, fontsize=9, fontweight='bold')
        ax.set_xlabel('Score' if row_idx == 1 else '')
        ax.set_ylabel('Count' if col_idx == 0 else '')
        ax.legend(loc='upper right', fontsize=8, frameon=True)

for col_idx in range(3):
    axes[0, col_idx].set_xlim(0.0, 1.0)
    axes[1, col_idx].set_xlim(0.0, 1.0)

axes[0, 3].set_xlim(-1.0, 1.0)
axes[1, 3].set_xlim(-1.0, 1.0)

for ax in axes.flatten():
    ax.autoscale(enable=False, axis='x')

plt.tight_layout()
plt.show()

## Bivariate Analysis

### Does the sentiment of feedback differ depending on which department is submitting feedback?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Select required columns
df_current = df[['department', 'Organization_rating', 'positive_feedback_EN_compound', 'negative_feedback_EN_compound']].copy()

# 1. Bin both positive and negative compound scores
sentiment_labels = ['Unfavorable', 'Indifferent', 'Favourable']
sentiment_bins = [-np.inf, -0.05, 0.05, np.inf]

df_current['positive_feedback_sentiment_class'] = pd.cut(
    df_current['positive_feedback_EN_compound'],
    bins=sentiment_bins,
    labels=sentiment_labels
)

df_current['negative_feedback_sentiment_class'] = pd.cut(
    df_current['negative_feedback_EN_compound'],
    bins=sentiment_bins,
    labels=sentiment_labels
)

# 2. Compute Crosstabs (percentages by department)
ct_positive = pd.crosstab(
    df_current['department'], 
    df_current['positive_feedback_sentiment_class'], 
    normalize='index'
) * 100

ct_negative = pd.crosstab(
    df_current['department'], 
    df_current['negative_feedback_sentiment_class'], 
    normalize='index'
) * 100

print("--- Positive Feedback Sentiment Distribution (%) ---")
print(ct_positive.round(2))
print("\n--- Negative Feedback Sentiment Distribution (%) ---")
print(ct_negative.round(2))

# 3. Plotting on 2 side-by-side subplots
colors = ['#e74c3c', '#95a5a6', '#2ecc71']  # Red (Neg), Grey (Neu), Green (Pos)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Tile 1: Positive Feedback Sentiment
ct_positive.plot(
    kind='bar', 
    stacked=True, 
    color=colors, 
    ax=axes[0], 
    legend=False
)
axes[0].set_title('Positive Feedback Sentiment by Department', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Department', fontsize=10)
axes[0].set_ylabel('Percentage (%)', fontsize=10)
axes[0].tick_params(axis='x', rotation=45)

# Tile 2: Negative Feedback Sentiment
ct_negative.plot(
    kind='bar', 
    stacked=True, 
    color=colors, 
    ax=axes[1], 
    legend=False
)
axes[1].set_title('Negative Feedback Sentiment by Department', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Department', fontsize=10)
axes[1].tick_params(axis='x', rotation=45)

# Single shared legend positioned outside the subplots
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Sentiment', bbox_to_anchor=(1.02, 0.5), loc='center left')

plt.tight_layout()
plt.show()

Note: It's worth looking at the negative feedbacks of the Technical department as they seem to be overly favourable.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

AXIS_MARGIN: float = .05
category_col = 'department' 
float_cols = df.select_dtypes(include=['float32'])

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()  
for i, col in enumerate(float_cols):
    sns.boxplot(
        data=df,
        x=category_col,
        y=col,
        ax=axes[i],
        # palette='Set2'
    )
    axes[i].set_title(f'{col} by {category_col}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('') 
    axes[i].set_ylabel('Value')
    axes[i].tick_params(axis='x', rotation=30)  
    axes[i].set(ylim=(-(1+AXIS_MARGIN), (1+AXIS_MARGIN)) if col.endswith('compound') else (-(0+AXIS_MARGIN), (1+AXIS_MARGIN)))


plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.diagnostic import het_breuschpagan

current_dfs = df.select_dtypes(include=['float32', 'category']).drop(labels=['survey_type', 'onboarding_rating'], axis=1)#.groupby(category_col)
# current_dfs.mean()

results = []
for col in float_cols:
    model = ols(f'{col} ~ C({category_col})', data=current_dfs).fit()
    residuals = model.resid
    shapiro_stat, shapiro_p = stats.shapiro(residuals)

    groups = [group[col].dropna() for _, group in df.groupby(category_col)]
    levene_stat, levene_p = stats.levene(*groups)
    
    results.append({
        'Variable': col,
        'Shapiro p-value': round(shapiro_p, 4),
        'Normality Passed': shapiro_p > 0.05,
        'Levene p-value': round(levene_p, 4),
        'Homogeneity Passed': levene_p > 0.05
    })

summary_df = pd.DataFrame(results)
summary_df

Due to failed tests for normality and small and unequal group sizes (3-32), ANOVA cannot be performed. Kruskal-Wallis test will be performed instead.

In [ ]:
import pandas as pd
from scipy import stats

def effect_size_mapping(eta_sq: float):
    if eta_sq < .01: return 'Negligible'
    if eta_sq < .06: return 'Tiny'
    if eta_sq < .14: return 'Medium'
    return 'Large'

results = []
for col in float_cols:

    groups = [group[col].dropna() for _, group in df.groupby(category_col)]
    k = len(groups)
    N = len(df)

    h_stat, p_val = stats.kruskal(*groups)

    eta_sq = (h_stat - k + 1) / (N - k)
    eta_sq_clamped = max(0.0, eta_sq)

    results.append({
        'Variable': col,
        'N (Total)': N,
        'k (Groups)': k,
        'H-statistic': round(h_stat, 4),
        'Eta-squared (η²)': round(eta_sq_clamped, 4),
        'Effect size': effect_size_mapping(round(eta_sq_clamped, 4)),
        'p-value': round(p_val, 4),
        'Significant': p_val < 0.05
    })

results_df = pd.DataFrame(results)
results_df

No variable has been deemed significant nor any effect bigger than 'Tiny'. Due to small group sizes, the test may be unable to detect subtle differences that actually exist in the population (Type II error). For now, these results suggest that all departments exhibit comparable baseline levels across all eight continuous factors, most importantly the compund sentiment scores.

This means that probably the departmental boundaries do not explain the variance in opinion scores in the "Survey 20 Days with SOP". No single department stands out as expressing systematically worse or better opinions than the others. Sentiment is uniformly distributed across the organization. If leadership is considering tailored policy changes, training, or communications department-by-department based on opinion scores, this data shows that approach is unwarranted. Interventions aimed at individual departments will likely address symptoms that are actually systemic across the entire organization.

### How does the distribution of onboarding_rating vary across departments?

Comparing the % value (instead of total counts) to get a fair comparisson between departments - reminder: we have almolt 2x more answers for 'O' than we have for 'P'.

In [ ]:
df_current = df.dropna(subset=['department', 'onboarding_rating']).copy()
df_current = df_current.groupby('department').filter(lambda x: len(x) >= 5)
ct = pd.crosstab(df_current['department'], df_current['onboarding_rating'], normalize='index')

sns.heatmap(ct, annot=True, fmt='.1%', cmap='Blues')
plt.title("Proportion of Onboarding Ratings by Department")
plt.show()

In [ ]:
import pandas as pd
from scipy import stats

STANDARD_SIGNIFICANCE_THRESHOLD = 0.05

df['onboarding_rating'] = df['onboarding_rating'].astype(float)
groups = [group['onboarding_rating'].values for _, group in df.groupby('department')]

h_stat, p_value = stats.kruskal(*groups)

print(f'H-statistic: {h_stat:.4f}')
print(f'p-value: {p_value:.4f}')
if p_value < STANDARD_SIGNIFICANCE_THRESHOLD:
    print('Result: Statistically significant difference found! At least one department has a different rating distribution.')
else:
    print('Result: No significant difference. Department ratings come from similar distributions.') # the differences between plots are likely random variation

## Time series analysis

### How does the participation of departments look over time?

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

df["posting_timestamp"] = pd.to_datetime(df["posting_timestamp"])
df["date"] = df["posting_timestamp"].dt.date

daily_counts = (
    df.groupby(["date", "department"]).size().unstack(fill_value=0)
)

full_date_range = pd.date_range(
    start=daily_counts.index.min(), end=daily_counts.index.max(), freq="D"
)
daily_counts = daily_counts.reindex(full_date_range, fill_value=0)

cumulative_counts = daily_counts.cumsum()
cumulative_prop = cumulative_counts.div(cumulative_counts.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(14, 7))
cumulative_prop.plot.area(
    ax=ax, stacked=True, colormap="tab10", alpha=0.85
)

label_dates = cumulative_prop.index[
    (cumulative_prop.index.day == 1)
    & (cumulative_prop.index.month.isin([1, 3, 5, 7, 9, 11]))
]

running_sum_prop = cumulative_prop.cumsum(axis=1)

for d in label_dates:
    if d not in cumulative_prop.index:
        continue

    bottom = 0
    for i, col in enumerate(cumulative_prop.columns):
        val = cumulative_prop.loc[d, col]

        # Only label if the category is thick enough (> 3% width) to avoid clutter
        if val > 0.03:
            top = running_sum_prop.loc[d, col]
            midpoint = bottom + (val / 2)

            # Format percentage string
            pct_text = f"{val*100:.0f}%"

            ax.text(
                pd.to_datetime(d),
                midpoint,
                pct_text,
                ha="center",
                va="center",
                fontsize=8,
                color="white",
                fontweight="bold",
            )

        bottom = running_sum_prop.loc[d, col]

plt.title(
    "Cumulative Department Proportion Over Time",
    fontsize=14,
    fontweight="bold",
)
plt.ylabel("Cumulative Share", fontsize=12)
plt.xlabel("Date", fontsize=12)

# Format y-axis as percentage
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Place legend outside
plt.legend(title="Department", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### How do the sentiment scores evolve over time?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Ensure datetime format
df["posting_timestamp"] = pd.to_datetime(df["posting_timestamp"])
df["date"] = df["posting_timestamp"].dt.date

# 2. Resample daily across the entire company (no department split)
daily_sentiment = (
    df.groupby("date")[
        ["positive_feedback_EN_compound", "negative_feedback_EN_compound"]
    ]
    .mean()
)

# Fill missing calendar days to maintain an accurate 30-day rolling window
full_date_range = pd.date_range(
    start=daily_sentiment.index.min(), end=daily_sentiment.index.max(), freq="D"
)
daily_long = daily_sentiment.reindex(full_date_range).reset_index()

# Fix for reset_index name
daily_long.rename(columns={"level_0": "date", "index": "date"}, inplace=True)

# 3. Calculate 30-day Moving Averages
daily_long["positive_30d_sma"] = (
    daily_long["positive_feedback_EN_compound"]
    .rolling(window=30, min_periods=1)
    .mean()
)
daily_long["negative_30d_sma"] = (
    daily_long["negative_feedback_EN_compound"]
    .rolling(window=30, min_periods=1)
    .mean()
)

# ==========================================
# PLOT 1: Positive Feedback (30-Day Moving Average)
# ==========================================
fig, ax1 = plt.subplots(figsize=(12, 5))

# Raw daily mean (faded)
sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_feedback_EN_compound",
    color="lightgreen",
    alpha=0.35,
    label="Daily Average",
    ax=ax1,
)

# 30-Day SMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_30d_sma",
    color="#1e8449",
    linewidth=2.5,
    label="30-Day Moving Avg",
    ax=ax1,
)

ax1.set_title(
    "Positive Feedback Sentiment: 30-Day Trend", fontsize=13, fontweight="bold"
)
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("VADER Compound Score", fontsize=11)
ax1.set_ylim(-1.05, 1.05)
ax1.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax1.tick_params(axis="x", rotation=30)
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()


# ==========================================
# PLOT 2: Negative Feedback (30-Day Moving Average)
# ==========================================
fig, ax2 = plt.subplots(figsize=(12, 5))

# Raw daily mean (faded)
sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_feedback_EN_compound",
    color="salmon",
    alpha=0.35,
    label="Daily Average",
    ax=ax2,
)

# 30-Day SMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_30d_sma",
    color="#78281f",
    linewidth=2.5,
    label="30-Day Moving Avg",
    ax=ax2,
)

ax2.set_title(
    "Negative Feedback Sentiment: 30-Day Trend", fontsize=13, fontweight="bold"
)
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("VADER Compound Score", fontsize=11)
ax2.set_ylim(-1.05, 1.05)
ax2.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax2.tick_params(axis="x", rotation=30)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()

Due to low answers volume and short overall time period, the simple moving average is chaotic for both feedback sentiment scores.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Prepare base DataFrame and Datetime columns
df_clean = df.dropna(subset=["posting_timestamp"]).copy()
df_clean["posting_timestamp"] = pd.to_datetime(df_clean["posting_timestamp"])
df_clean["date"] = df_clean["posting_timestamp"].dt.date
df_clean["Year-Month"] = df_clean["posting_timestamp"].dt.strftime("%Y-%m")

# 2. Compute Monthly Post Counts
monthly_counts = df_clean.groupby("Year-Month").size().to_dict()

# 3. Resample daily across full date range
daily_sentiment = (
    df_clean.groupby("date")[
        ["positive_feedback_EN_compound", "negative_feedback_EN_compound"]
    ]
    .mean()
)

full_date_range = pd.date_range(
    start=daily_sentiment.index.min(), end=daily_sentiment.index.max(), freq="D"
)
daily_long = daily_sentiment.reindex(full_date_range).reset_index()
daily_long.rename(columns={"level_0": "date", "index": "date"}, inplace=True)

daily_long["Year-Month"] = daily_long["date"].dt.strftime("%Y-%m")
daily_long["monthly_post_count"] = daily_long["Year-Month"].map(monthly_counts).fillna(0)

# 4. Calculate 30-day Exponential Moving Averages (EMA)
daily_long["positive_30d_ema"] = (
    daily_long["positive_feedback_EN_compound"]
    .ewm(span=30, adjust=False)
    .mean()
)
daily_long["negative_30d_ema"] = (
    daily_long["negative_feedback_EN_compound"]
    .ewm(span=60, adjust=False)
    .mean()
)


# ==========================================
# PLOT 1: Positive Feedback (30-Day EMA) + Monthly Volume
# ==========================================
fig, ax1 = plt.subplots(figsize=(13, 6))

# Secondary Y-Axis for Volume
ax1_twin = ax1.twinx()
ax1_twin.grid(False)
ax1_twin.bar(
    daily_long["date"],
    daily_long["monthly_post_count"],
    color="#3498db",
    alpha=0.18,
    width=1.0,
    label="Monthly Post Volume",
)
ax1_twin.set_ylabel("Monthly Post Count", color="#2980b9", fontsize=11)
ax1_twin.tick_params(axis="y", labelcolor="#2980b9")

# Primary Y-Axis: Raw Daily vs. 30-Day EMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_feedback_EN_compound",
    color="lightgreen",
    alpha=0.35,
    label="Daily Avg Score",
    ax=ax1,
)

sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_30d_ema",
    color="#1e8449",
    linewidth=2.5,
    label="30-Day Exponential Moving Avg (EMA)",
    ax=ax1,
)

ax1.set_title(
    "Positive Feedback Sentiment (30-Day EMA) & Monthly Volume",
    fontsize=13,
    fontweight="bold",
)
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("VADER Compound Score", fontsize=11)
ax1.set_ylim(-1.05, 1.05)
ax1.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax1.tick_params(axis="x", rotation=30)
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()


# ==========================================
# PLOT 2: Negative Feedback (30-Day EMA) + Monthly Volume
# ==========================================
fig, ax2 = plt.subplots(figsize=(13, 6))

# Secondary Y-Axis for Volume
ax2_twin = ax2.twinx()
ax2_twin.grid(False)
ax2_twin.bar(
    daily_long["date"],
    daily_long["monthly_post_count"],
    color="#3498db",
    alpha=0.18,
    width=1.0,
    label="Monthly Post Volume",
)
ax2_twin.set_ylabel("Monthly Post Count", color="#2980b9", fontsize=11)
ax2_twin.tick_params(axis="y", labelcolor="#2980b9")

# Primary Y-Axis: Raw Daily vs. 30-Day EMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_feedback_EN_compound",
    color="salmon",
    alpha=0.35,
    label="Daily Avg Score",
    ax=ax2,
)

sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_30d_ema",
    color="#78281f",
    linewidth=2.5,
    label="30-Day Exponential Moving Avg (EMA)",
    ax=ax2,
)

ax2.set_title(
    "Negative Feedback Sentiment (30-Day EMA) & Monthly Volume",
    fontsize=13,
    fontweight="bold",
)
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("VADER Compound Score", fontsize=11)
ax2.set_ylim(-1.05, 1.05)
ax2.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax2.tick_params(axis="x", rotation=30)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()

With some degree of certainty we can claim an improvement in the negative feedback sentiment score over time. However the same cannot be said about the positive feedback sentiment score, as it mostly varies around the value of 0.5.
We might also be seeing long-term cyclic behaviour forming - more data needed to test this hypothesis. The current sentiment analysis sugests that the year-long period between April 2025 and April 2026 were very well received by the volunteers.

### How does the onboarding rating evolve over time?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

df["onboarding_rating"] = df["onboarding_rating"].astype(float)
df_clean = df.dropna(subset=["posting_timestamp"]).copy()
df_clean["posting_timestamp"] = pd.to_datetime(df_clean["posting_timestamp"])
df_clean["date"] = df_clean["posting_timestamp"].dt.date
df_clean["Year-Month"] = df_clean["posting_timestamp"].dt.strftime("%Y-%m")

monthly_counts = df_clean.groupby("Year-Month").size().to_dict()

daily_rating = df_clean.groupby("date")["onboarding_rating"].mean()

full_date_range = pd.date_range(
    start=daily_rating.index.min(), end=daily_rating.index.max(), freq="D"
)
daily_df = daily_rating.reindex(full_date_range).reset_index()
daily_df.rename(columns={"level_0": "date", "index": "date"}, inplace=True)

daily_df["Year-Month"] = daily_df["date"].dt.strftime("%Y-%m")
daily_df["monthly_post_count"] = daily_df["Year-Month"].map(monthly_counts).fillna(0)

daily_df["rating_30d_ema"] = (
    daily_df["onboarding_rating"]
    .ewm(span=30, adjust=False)
    .mean()
)

fig, ax1 = plt.subplots(figsize=(13, 6))

ax2 = ax1.twinx()
ax2.grid(False)
ax2.bar(
    daily_df["date"],
    daily_df["monthly_post_count"],
    color="#3498db",
    alpha=0.18,
    width=1.0,
    label="Monthly Post Volume",
)
ax2.set_ylabel("Monthly Post Count", color="#2980b9", fontsize=11)
ax2.tick_params(axis="y", labelcolor="#2980b9")

sns.lineplot(
    data=daily_df,
    x="date",
    y="onboarding_rating",
    color="skyblue",
    alpha=0.4,
    label="Daily Avg Rating",
    ax=ax1,
)

sns.lineplot(
    data=daily_df,
    x="date",
    y="rating_30d_ema",
    color="#2980b9",
    linewidth=2.5,
    label="30-Day EMA",
    ax=ax1,
)

ax1.set_title(
    "Onboarding Rating (1-5 Scale): 30-Day EMA & Monthly Volume",
    fontsize=13,
    fontweight="bold",
)
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("Rating Scale (1 - 5)", fontsize=11)
ax1.set_ylim(1.0, 5.0)
ax1.axhline(3.0, color="gray", linestyle=":", alpha=0.6)
ax1.tick_params(axis="x", rotation=30)
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()

We see an unmistakable upward trend of onboarding rating over time. We do not see any operational bottlenecks - the trend is upwards irregardless of dynamic changes in the montly post count.

# Survey 6 Months with SOP

In [ ]:
current_data_period: DataFilePeriod = DataFilePeriod.HALF_YEAR

df = inputs.get_input_data(preferred_file_type, current_data_period)
df.insert(0, 'posting_timestamp', df['posting_day'] + ' ' + df['posting_hour'])
df.drop(['posting_day', 'posting_hour'], axis=1, inplace=True)

strategy: PreprocessingStrategy = PreprocessingStrategy(
    type_map={
        'positive_feedback': 'string[pyarrow]', 
        'negative_feedback': 'string[pyarrow]', 
        'positive_feedback_EN': 'string[pyarrow]', 
        'negative_feedback_EN': 'string[pyarrow]',
        'positive_feedback_EN_neg': 'float32',
        'positive_feedback_EN_neu': 'float32',
        'positive_feedback_EN_pos': 'float32',
        'positive_feedback_EN_compound': 'float32',
        'negative_feedback_EN_neg': 'float32',
        'negative_feedback_EN_neu': 'float32',
        'negative_feedback_EN_pos': 'float32',
        'negative_feedback_EN_compound': 'float32',
    },
    datetime_cols=['posting_timestamp'],
    categorical_cols=['survey_type'],
    ordinal_cols={'organization_rating': list(range(1,6))}
)

preprocessor: DataPreprocessor = DataPreprocessor(df)
preprocessor.set_dtypes(strategy)
df = preprocessor._df
df

## Univariate Analysis

In [ ]:
print(df.shape)
print('---\n')
print(df.info())
print('---\n')
print(df.head())
print('---\n')
print('The indicies are unique: ', df.index.nunique() == len(df))

### Datetime variables

Datetime variables require specialized analysis because time functions both as an index (ordering events sequentially) and a feature (reflecting cyclical human behavior and seasonality).
The main goal of datetime univariate analysis is to evaluate data continuity, spot missing time gaps, assess time resolution, and understand periodic patterns.

#### posting_timestamp

In [ ]:
current_series = df['posting_timestamp']
t_min, t_max = current_series.min(), current_series.max()
total_span = (t_max - t_min)
time_deltas = current_series.diff()

def td_format(td_object):
    seconds = int(td_object.total_seconds())
    periods = [
        ('year',        60*60*24*365),
        ('month',       60*60*24*30),
        ('day',         60*60*24),
        ('hour',        60*60),
        ('minute',      60),
        ('second',      1)
    ]

    strings=[]
    for period_name, period_seconds in periods:
        if seconds > period_seconds:
            period_value , seconds = divmod(seconds, period_seconds)
            has_s = 's' if period_value > 1 else ''
            strings.append("%s %s%s" % (period_value, period_name, has_s))

    return " ".join(strings)

print(f'''
Temporal coverage
    t_min: {t_min:%Y/%m/%d}
    t_max: {t_max:%Y/%m/%d}
    total_span: {td_format(total_span)}
---
Sampling Interval & Resolution
    median difference between answer times: {td_format(time_deltas.median())}
    average difference between answer times: {td_format(time_deltas.mean())}
''')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_datetime = df.dropna(subset=['posting_timestamp']).copy()

df_datetime['Year-Month'] = [ts.strftime('%Y-%m') for ts in df_datetime['posting_timestamp']]
df_datetime['DayOfWeek']  = [ts.strftime('%a') for ts in df_datetime['posting_timestamp']]
df_datetime['Hour']       = [ts.hour for ts in df_datetime['posting_timestamp']]

day_order = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

monthly_map = {}
for ym in df_datetime['Year-Month']:
    monthly_map[ym] = monthly_map.get(ym, 0) + 1

if monthly_map:
    min_ym = min(monthly_map.keys())
    max_ym = max(monthly_map.keys())
    
    start_y, start_m = map(int, min_ym.split('-'))
    end_y, end_m     = map(int, max_ym.split('-'))
    
    all_months = []
    curr_y, curr_m = start_y, start_m
    while (curr_y, curr_m) <= (end_y, end_m):
        all_months.append(f"{curr_y:04d}-{curr_m:02d}")
        curr_m += 1
        if curr_m > 12:
            curr_m = 1
            curr_y += 1
            
    monthly_keys = all_months
    monthly_vals = [monthly_map.get(ym, 0) for ym in monthly_keys]
else:
    monthly_keys = []
    monthly_vals = []

daily_map = {}
for d in df_datetime['DayOfWeek']:
    daily_map[d] = daily_map.get(d, 0) + 1

daily_vals = [daily_map.get(d, 0) for d in day_order]

hourly_map = {}
for h in df_datetime['Hour']:
    hourly_map[h] = hourly_map.get(h, 0) + 1

hourly_vals = [hourly_map.get(h, 0) for h in range(24)]

y_max = max(max(monthly_vals, default=0), max(daily_vals, default=0), max(hourly_vals, default=0)) * 1.15

fig, axes = plt.subplots(
    1, 3, 
    figsize=(18, 5), 
    sharey=True, 
    gridspec_kw={'width_ratios': [1.8, 0.6, 1.6]}
)

sns.barplot(
    x=monthly_keys,
    y=monthly_vals,
    hue=monthly_keys,
    legend=False,
    ax=axes[0],
    palette='Blues_d'
)
axes[0].set_title('Post Frequency by Month')
axes[0].set_ylabel('Number of Posts')
axes[0].set_xlabel('Year-Month')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(
    x=day_order, 
    y=daily_vals, 
    hue=day_order, 
    legend=False, 
    ax=axes[1], 
    palette='Blues_d'
)
axes[1].set_title('Post Frequency by Day')
axes[1].set_xlabel('DayOfWeek')
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(
    x=list(range(24)), 
    y=hourly_vals, 
    hue=list(range(24)), 
    legend=False, 
    ax=axes[2], 
    palette='viridis'
)
axes[2].set_title('Post Frequency by Hour of Day')
axes[2].set_xlabel('Hour')
axes[2].set_xticks([i - 0.5 for i in range(25)])
axes[2].set_xticklabels([f"{h:02d}:00" for h in range(25)], rotation=45)

for ax in axes:
    for container in ax.containers:
        labels = [f'{int(v)}' if v > 0 else '' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=2, fontsize=8)

axes[0].set_ylim(0, y_max)

plt.tight_layout()
plt.show()

Note: the y-axis is common for all 3 plots. Additionally, given the intuition of something happening IN a given month or ON a given day of the week, the tickmarks under the first 2 plots are directly under the bars. However, the same cannot be said about hours, therefore, the tickmarks were explicitly shifted to highlight that the bar in between, say, 14:00 and 15:00 ticks show counts for posts in time range [14:00, 15:00).

Note: by "posting" here we mean the event of a feedback message being visible to the members of channel #15-feedback

In [ ]:
df_datetime = df.dropna(subset=['posting_timestamp']).copy()

df_datetime['DayOfWeek'] = df_datetime['posting_timestamp'].dt.day_name()
labels = ['00:00-03:59', '04:00-07:59', '08:00-11:59', '12:00-15:59', '16:00-19:59', '20:00-23:59']

df_datetime['Hour_4H'] = pd.cut(
    df_datetime['posting_timestamp'].dt.hour, 
    bins=[-1, 3, 7, 11, 15, 19, 23], 
    labels=labels
)

day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

heatmap_data = pd.crosstab(
    df_datetime['DayOfWeek'], 
    df_datetime['Hour_4H']
).reindex(day_order)

plt.figure(figsize=(12, 5))
sns.heatmap(heatmap_data, cmap='YlGnBu', annot=True,  cbar_kws={'label': 'Post Count'})
plt.title('Post Frequency Heatmap (Day of Week vs. Hour of Day)')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

The start of the week has been set to Sunday to maintain the comparability with the previous section.

### Nominal variables

Nominal variables are such categorical variables where categories have no inherent order or ranking.
The univariate analysis here focuses on frequency distributions, central tendency (mode), and cardinality/entropy.

#### survey_type

Just sanity checks, this variable here is not necessary from the point of view of the decision making. We expect it to be not null and constant everywhere; `survey_type='6 Month'`.

In [ ]:
st = df['survey_type']
is_valid = (st.eq('6 Month')).all()
print(f'''
The variable survey_type is valid: {is_valid}
''')

### Ordinal variables

#### organization_rating

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

current_series = df['organization_rating']
total_obs = len(current_series)
ordinal_order = list(current_series.dtype.categories)

freq_map = {}
for item in current_series:
    key = 'Unspecified' if item is None or (hasattr(item, 'isna') and item.isna()) or str(item) in ('nan', 'Unspecified') else str(item)
    freq_map[key] = freq_map.get(key, 0) + 1

categories = [str(cat) for cat in ordinal_order] + ['Unspecified']
counts = [freq_map.get(cat, 0) for cat in categories]

fig, ax = plt.subplots(figsize=(8, 4.5))

palette = ['gray' if cat == 'Unspecified' else 'tab:blue' for cat in categories]

sns.barplot(
    x=categories,
    y=counts,
    hue=categories,
    legend=False,
    ax=ax,
    palette=palette
)

ax.set_title('Organization Rating Frequency Distribution')
ax.set_xlabel('Organization Rating')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)

for container in ax.containers:
    labels = [f'{int(v)}\n({v/total_obs:.2%})' for v in container.datavalues]
    ax.bar_label(container, labels=labels, padding=3, fontsize=8)

ax.set_ylim(0, max(counts, default=1) * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
*ordered_valid, na_counter = list(zip(categories, counts))
na_count = na_counter[1]
average_score = sum(int(cat) * count for (cat, count) in ordered_valid)/total_obs
median_score = current_series.cat.codes[current_series.cat.codes >= 0].median()
mode_key, mode_val = max(ordered_valid, key=lambda x: x[1])
antimode_key, antimode_val = min(ordered_valid, key=lambda x: x[1])


print(f'''
The average score is "{average_score:.2f}".
The median score is "{int(median_score) if int(median_score) == median_score else median_score}".
The mode score is "{mode_key}" with value {mode_val}, constituting therefore {mode_val/total_obs:.2%} of all observations.
The antimode score is "{antimode_key}" with value {antimode_val}, constituting therefore {antimode_val/total_obs:.2%} of all observations.
There are {na_count} answer{'s' if na_count != 1 else ''} with no score specified, constituting therefore {na_count/total_obs:.2%} of all observations.
''')

average_score

### Free text variables

Univariate analysis of free-text (unstructured text) data focuses on summarizing the distribution, statistical properties, length characteristics, and dominant vocabulary or themes of a single text column.

#### positive_feedback

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['positive_feedback']
lang = 'PL'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', '<NA>'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

Lexical Richness (or Lexical Diversity) evaluates the variety and breadth of vocabulary used in a text corpus relative to its total length. Here measured by Corpus and Mean Document-Level TTR.

Type-Token Ratio (TTR):
$$\text{TTR} = \frac{\text{Unique Words (Types)}}{\text{Total Words (Tokens)}}$$

Scale: $0.0$ to $1.0$.

Interpretation: Higher values indicate richer vocabulary. 

However, TTR naturally drops as text length increases because common words repeat.


In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.8, not 0.0.

Note: CTTR < MDLTTR => Feedback is consise (volunteers don't write much) and focused (repeated topics).

#### negative_feedback

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['negative_feedback']
lang = 'PL'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', 'Unspecified'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.8, not 0.0.

Note: CTTR < MDLTTR => Feedback is consise (volunteers don't write much) and focused (repeated topics).

#### positive_feedback_EN

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['positive_feedback_EN']
lang = 'EN'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', '<NA>'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.6, not 0.0.

Note: CTTR < MDLTTR => Feedback is consise (volunteers don't write much) and focused (repeated topics).

#### negative_feedback_EN

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['negative_feedback_EN']
lang = 'EN'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', '<NA>'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.6, not 0.0.

Note: CTTR < MDLTTR => Feedback is consise (volunteers don't write much) and focused (repeated topics).

### Continuous variables

When performing univariate analysis on continuous variables, the focus shifts to distribution shape, central tendency, spread, tail behavior, and data integrity.

#### Sentiment variables: [positive|negative]\_feedback_EN\_[neg|neu|pos|compound]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

grid_layout = [
    ['positive_feedback_EN_neg', 'positive_feedback_EN_neu', 'positive_feedback_EN_pos', 'positive_feedback_EN_compound'],
    ['negative_feedback_EN_neg', 'negative_feedback_EN_neu', 'negative_feedback_EN_pos', 'negative_feedback_EN_compound']
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True, sharex='col')

for row_idx in range(2):
    for col_idx in range(4):
        ax = axes[row_idx, col_idx]
        col_name = grid_layout[row_idx][col_idx]
        current_series = df[col_name].dropna()
        
        mean_val = current_series.mean() if not current_series.empty else 0
        median_val = current_series.median() if not current_series.empty else 0
        
        sns.histplot(current_series, kde=True, ax=ax, color='tab:blue', bins=30)
        
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.2, label=f'Mean: {mean_val:.2f}')
        ax.axvline(median_val, color='green', linestyle='-', linewidth=1.2, label=f'Med: {median_val:.2f}')
        
        ax.set_title(col_name, fontsize=9, fontweight='bold')
        ax.set_xlabel('Score' if row_idx == 1 else '')
        ax.set_ylabel('Count' if col_idx == 0 else '')
        ax.legend(loc='upper right', fontsize=8, frameon=True)

for col_idx in range(3):
    axes[0, col_idx].set_xlim(0.0, 1.0)
    axes[1, col_idx].set_xlim(0.0, 1.0)

axes[0, 3].set_xlim(-1.0, 1.0)
axes[1, 3].set_xlim(-1.0, 1.0)

for ax in axes.flatten():
    ax.autoscale(enable=False, axis='x')

plt.tight_layout()
plt.show()

## Bivariate Analysis

## Time series analysis

### How do the sentiment scores evolve over time?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Ensure datetime format
df["posting_timestamp"] = pd.to_datetime(df["posting_timestamp"])
df["date"] = df["posting_timestamp"].dt.date

# 2. Resample daily across the entire company (no department split)
daily_sentiment = (
    df.groupby("date")[
        ["positive_feedback_EN_compound", "negative_feedback_EN_compound"]
    ]
    .mean()
)

# Fill missing calendar days to maintain an accurate 30-day rolling window
full_date_range = pd.date_range(
    start=daily_sentiment.index.min(), end=daily_sentiment.index.max(), freq="D"
)
daily_long = daily_sentiment.reindex(full_date_range).reset_index()

# Fix for reset_index name
daily_long.rename(columns={"level_0": "date", "index": "date"}, inplace=True)

# 3. Calculate 30-day Moving Averages
daily_long["positive_30d_sma"] = (
    daily_long["positive_feedback_EN_compound"]
    .rolling(window=30, min_periods=1)
    .mean()
)
daily_long["negative_30d_sma"] = (
    daily_long["negative_feedback_EN_compound"]
    .rolling(window=30, min_periods=1)
    .mean()
)

# ==========================================
# PLOT 1: Positive Feedback (30-Day Moving Average)
# ==========================================
fig, ax1 = plt.subplots(figsize=(12, 5))

# Raw daily mean (faded)
sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_feedback_EN_compound",
    color="lightgreen",
    alpha=0.35,
    label="Daily Average",
    ax=ax1,
)

# 30-Day SMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_30d_sma",
    color="#1e8449",
    linewidth=2.5,
    label="30-Day Moving Avg",
    ax=ax1,
)

ax1.set_title(
    "Positive Feedback Sentiment: 30-Day Trend", fontsize=13, fontweight="bold"
)
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("VADER Compound Score", fontsize=11)
ax1.set_ylim(-1.05, 1.05)
ax1.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax1.tick_params(axis="x", rotation=30)
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()


# ==========================================
# PLOT 2: Negative Feedback (30-Day Moving Average)
# ==========================================
fig, ax2 = plt.subplots(figsize=(12, 5))

# Raw daily mean (faded)
sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_feedback_EN_compound",
    color="salmon",
    alpha=0.35,
    label="Daily Average",
    ax=ax2,
)

# 30-Day SMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_30d_sma",
    color="#78281f",
    linewidth=2.5,
    label="30-Day Moving Avg",
    ax=ax2,
)

ax2.set_title(
    "Negative Feedback Sentiment: 30-Day Trend", fontsize=13, fontweight="bold"
)
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("VADER Compound Score", fontsize=11)
ax2.set_ylim(-1.05, 1.05)
ax2.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax2.tick_params(axis="x", rotation=30)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()

Due to low answers volume and short overall time period, the simple moving average is chaotic for both feedback sentiment scores.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Prepare base DataFrame and Datetime columns
df_clean = df.dropna(subset=["posting_timestamp"]).copy()
df_clean["posting_timestamp"] = pd.to_datetime(df_clean["posting_timestamp"])
df_clean["date"] = df_clean["posting_timestamp"].dt.date
df_clean["Year-Month"] = df_clean["posting_timestamp"].dt.strftime("%Y-%m")

# 2. Compute Monthly Post Counts
monthly_counts = df_clean.groupby("Year-Month").size().to_dict()

# 3. Resample daily across full date range
daily_sentiment = (
    df_clean.groupby("date")[
        ["positive_feedback_EN_compound", "negative_feedback_EN_compound"]
    ]
    .mean()
)

full_date_range = pd.date_range(
    start=daily_sentiment.index.min(), end=daily_sentiment.index.max(), freq="D"
)
daily_long = daily_sentiment.reindex(full_date_range).reset_index()
daily_long.rename(columns={"level_0": "date", "index": "date"}, inplace=True)

daily_long["Year-Month"] = daily_long["date"].dt.strftime("%Y-%m")
daily_long["monthly_post_count"] = daily_long["Year-Month"].map(monthly_counts).fillna(0)

# 4. Calculate 30-day Exponential Moving Averages (EMA)
daily_long["positive_30d_ema"] = (
    daily_long["positive_feedback_EN_compound"]
    .ewm(span=30, adjust=False)
    .mean()
)
daily_long["negative_30d_ema"] = (
    daily_long["negative_feedback_EN_compound"]
    .ewm(span=60, adjust=False)
    .mean()
)


# ==========================================
# PLOT 1: Positive Feedback (30-Day EMA) + Monthly Volume
# ==========================================
fig, ax1 = plt.subplots(figsize=(13, 6))

# Secondary Y-Axis for Volume
ax1_twin = ax1.twinx()
ax1_twin.grid(False)
ax1_twin.bar(
    daily_long["date"],
    daily_long["monthly_post_count"],
    color="#3498db",
    alpha=0.18,
    width=1.0,
    label="Monthly Post Volume",
)
ax1_twin.set_ylabel("Monthly Post Count", color="#2980b9", fontsize=11)
ax1_twin.tick_params(axis="y", labelcolor="#2980b9")

# Primary Y-Axis: Raw Daily vs. 30-Day EMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_feedback_EN_compound",
    color="lightgreen",
    alpha=0.35,
    label="Daily Avg Score",
    ax=ax1,
)

sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_30d_ema",
    color="#1e8449",
    linewidth=2.5,
    label="30-Day Exponential Moving Avg (EMA)",
    ax=ax1,
)

ax1.set_title(
    "Positive Feedback Sentiment (30-Day EMA) & Monthly Volume",
    fontsize=13,
    fontweight="bold",
)
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("VADER Compound Score", fontsize=11)
ax1.set_ylim(-1.05, 1.05)
ax1.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax1.tick_params(axis="x", rotation=30)
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()


# ==========================================
# PLOT 2: Negative Feedback (30-Day EMA) + Monthly Volume
# ==========================================
fig, ax2 = plt.subplots(figsize=(13, 6))

# Secondary Y-Axis for Volume
ax2_twin = ax2.twinx()
ax2_twin.grid(False)
ax2_twin.bar(
    daily_long["date"],
    daily_long["monthly_post_count"],
    color="#3498db",
    alpha=0.18,
    width=1.0,
    label="Monthly Post Volume",
)
ax2_twin.set_ylabel("Monthly Post Count", color="#2980b9", fontsize=11)
ax2_twin.tick_params(axis="y", labelcolor="#2980b9")

# Primary Y-Axis: Raw Daily vs. 30-Day EMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_feedback_EN_compound",
    color="salmon",
    alpha=0.35,
    label="Daily Avg Score",
    ax=ax2,
)

sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_30d_ema",
    color="#78281f",
    linewidth=2.5,
    label="30-Day Exponential Moving Avg (EMA)",
    ax=ax2,
)

ax2.set_title(
    "Negative Feedback Sentiment (30-Day EMA) & Monthly Volume",
    fontsize=13,
    fontweight="bold",
)
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("VADER Compound Score", fontsize=11)
ax2.set_ylim(-1.05, 1.05)
ax2.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax2.tick_params(axis="x", rotation=30)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()

With some degree of certainty we can claim an improvement in the negative feedback sentiment score over time. However the same cannot be said about the positive feedback sentiment score, as it mostly varies around the value of 0.25. Interestingly enough, the negative feedback also hovers around the same value.

### How does the organization rating evolve over time?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

df["organization_rating"] = df["organization_rating"].astype(float)
df_clean = df.dropna(subset=["posting_timestamp"]).copy()
df_clean["posting_timestamp"] = pd.to_datetime(df_clean["posting_timestamp"])
df_clean["date"] = df_clean["posting_timestamp"].dt.date
df_clean["Year-Month"] = df_clean["posting_timestamp"].dt.strftime("%Y-%m")

monthly_counts = df_clean.groupby("Year-Month").size().to_dict()

daily_rating = df_clean.groupby("date")["organization_rating"].mean()

full_date_range = pd.date_range(
    start=daily_rating.index.min(), end=daily_rating.index.max(), freq="D"
)
daily_df = daily_rating.reindex(full_date_range).reset_index()
daily_df.rename(columns={"level_0": "date", "index": "date"}, inplace=True)

daily_df["Year-Month"] = daily_df["date"].dt.strftime("%Y-%m")
daily_df["monthly_post_count"] = daily_df["Year-Month"].map(monthly_counts).fillna(0)

daily_df["rating_30d_ema"] = (
    daily_df["organization_rating"]
    .ewm(span=30, adjust=False)
    .mean()
)

fig, ax1 = plt.subplots(figsize=(13, 6))

ax2 = ax1.twinx()
ax2.grid(False)
ax2.bar(
    daily_df["date"],
    daily_df["monthly_post_count"],
    color="#3498db",
    alpha=0.18,
    width=1.0,
    label="Monthly Post Volume",
)
ax2.set_ylabel("Monthly Post Count", color="#2980b9", fontsize=11)
ax2.tick_params(axis="y", labelcolor="#2980b9")

sns.lineplot(
    data=daily_df,
    x="date",
    y="organization_rating",
    color="skyblue",
    alpha=0.4,
    label="Daily Avg Rating",
    ax=ax1,
)

sns.lineplot(
    data=daily_df,
    x="date",
    y="rating_30d_ema",
    color="#2980b9",
    linewidth=2.5,
    label="30-Day EMA",
    ax=ax1,
)

ax1.set_title(
    "Organization Rating (1-5 Scale): 30-Day EMA & Monthly Volume",
    fontsize=13,
    fontweight="bold",
)
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("Rating Scale (1 - 5)", fontsize=11)
ax1.set_ylim(1.0, 5.0)
ax1.axhline(3.0, color="gray", linestyle=":", alpha=0.6)
ax1.tick_params(axis="x", rotation=30)
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()

We see a very stable trend of organization rating over time. We do not see any operational bottlenecks - the trend is stable irregardless of the changes in the montly post count.

# Survey 1 Year with SOP

In [ ]:
current_data_period: DataFilePeriod = DataFilePeriod.FULL_YEAR

df = inputs.get_input_data(preferred_file_type, current_data_period)
df.insert(0, 'posting_timestamp', df['posting_day'] + ' ' + df['posting_hour'])
df.drop(['posting_day', 'posting_hour'], axis=1, inplace=True)

strategy: PreprocessingStrategy = PreprocessingStrategy(
    type_map={
        'positive_feedback': 'string[pyarrow]', 
        'negative_feedback': 'string[pyarrow]', 
        'positive_feedback_EN': 'string[pyarrow]', 
        'negative_feedback_EN': 'string[pyarrow]',
        'positive_feedback_EN_neg': 'float32',
        'positive_feedback_EN_neu': 'float32',
        'positive_feedback_EN_pos': 'float32',
        'positive_feedback_EN_compound': 'float32',
        'negative_feedback_EN_neg': 'float32',
        'negative_feedback_EN_neu': 'float32',
        'negative_feedback_EN_pos': 'float32',
        'negative_feedback_EN_compound': 'float32',
    },
    datetime_cols=['posting_timestamp'],
    categorical_cols=['survey_type'],
    ordinal_cols={'organization_rating': list(range(1,6))}
)

preprocessor: DataPreprocessor = DataPreprocessor(df)
preprocessor.set_dtypes(strategy)
df = preprocessor._df
df

## Univariate Analysis

In [ ]:
print(df.shape)
print('---\n')
print(df.info())
print('---\n')
print(df.head())
print('---\n')
print('The indicies are unique: ', df.index.nunique() == len(df))

### Datetime variables

Datetime variables require specialized analysis because time functions both as an index (ordering events sequentially) and a feature (reflecting cyclical human behavior and seasonality).
The main goal of datetime univariate analysis is to evaluate data continuity, spot missing time gaps, assess time resolution, and understand periodic patterns.

#### posting_timestamp

In [ ]:
current_series = df['posting_timestamp']
t_min, t_max = current_series.min(), current_series.max()
total_span = (t_max - t_min)
time_deltas = current_series.diff()

def td_format(td_object):
    seconds = int(td_object.total_seconds())
    periods = [
        ('year',        60*60*24*365),
        ('month',       60*60*24*30),
        ('day',         60*60*24),
        ('hour',        60*60),
        ('minute',      60),
        ('second',      1)
    ]

    strings=[]
    for period_name, period_seconds in periods:
        if seconds > period_seconds:
            period_value , seconds = divmod(seconds, period_seconds)
            has_s = 's' if period_value > 1 else ''
            strings.append("%s %s%s" % (period_value, period_name, has_s))

    return " ".join(strings)

print(f'''
Temporal coverage
    t_min: {t_min:%Y/%m/%d}
    t_max: {t_max:%Y/%m/%d}
    total_span: {td_format(total_span)}
---
Sampling Interval & Resolution
    median difference between answer times: {td_format(time_deltas.median())}
    average difference between answer times: {td_format(time_deltas.mean())}
''')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_datetime = df.dropna(subset=['posting_timestamp']).copy()

df_datetime['Year-Month'] = [ts.strftime('%Y-%m') for ts in df_datetime['posting_timestamp']]
df_datetime['DayOfWeek']  = [ts.strftime('%a') for ts in df_datetime['posting_timestamp']]
df_datetime['Hour']       = [ts.hour for ts in df_datetime['posting_timestamp']]

day_order = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

monthly_map = {}
for ym in df_datetime['Year-Month']:
    monthly_map[ym] = monthly_map.get(ym, 0) + 1

if monthly_map:
    min_ym = min(monthly_map.keys())
    max_ym = max(monthly_map.keys())
    
    start_y, start_m = map(int, min_ym.split('-'))
    end_y, end_m     = map(int, max_ym.split('-'))
    
    all_months = []
    curr_y, curr_m = start_y, start_m
    while (curr_y, curr_m) <= (end_y, end_m):
        all_months.append(f"{curr_y:04d}-{curr_m:02d}")
        curr_m += 1
        if curr_m > 12:
            curr_m = 1
            curr_y += 1
            
    monthly_keys = all_months
    monthly_vals = [monthly_map.get(ym, 0) for ym in monthly_keys]
else:
    monthly_keys = []
    monthly_vals = []

daily_map = {}
for d in df_datetime['DayOfWeek']:
    daily_map[d] = daily_map.get(d, 0) + 1

daily_vals = [daily_map.get(d, 0) for d in day_order]

hourly_map = {}
for h in df_datetime['Hour']:
    hourly_map[h] = hourly_map.get(h, 0) + 1

hourly_vals = [hourly_map.get(h, 0) for h in range(24)]

y_max = max(max(monthly_vals, default=0), max(daily_vals, default=0), max(hourly_vals, default=0)) * 1.15

fig, axes = plt.subplots(
    1, 3, 
    figsize=(18, 5), 
    sharey=True, 
    gridspec_kw={'width_ratios': [1.8, 0.6, 1.6]}
)

sns.barplot(
    x=monthly_keys,
    y=monthly_vals,
    hue=monthly_keys,
    legend=False,
    ax=axes[0],
    palette='Blues_d'
)
axes[0].set_title('Post Frequency by Month')
axes[0].set_ylabel('Number of Posts')
axes[0].set_xlabel('Year-Month')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(
    x=day_order, 
    y=daily_vals, 
    hue=day_order, 
    legend=False, 
    ax=axes[1], 
    palette='Blues_d'
)
axes[1].set_title('Post Frequency by Day')
axes[1].set_xlabel('DayOfWeek')
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(
    x=list(range(24)), 
    y=hourly_vals, 
    hue=list(range(24)), 
    legend=False, 
    ax=axes[2], 
    palette='viridis'
)
axes[2].set_title('Post Frequency by Hour of Day')
axes[2].set_xlabel('Hour')
axes[2].set_xticks([i - 0.5 for i in range(25)])
axes[2].set_xticklabels([f"{h:02d}:00" for h in range(25)], rotation=45)

for ax in axes:
    for container in ax.containers:
        labels = [f'{int(v)}' if v > 0 else '' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=2, fontsize=8)

axes[0].set_ylim(0, y_max)

plt.tight_layout()
plt.show()

Note: the y-axis is common for all 3 plots. Additionally, given the intuition of something happening IN a given month or ON a given day of the week, the tickmarks under the first 2 plots are directly under the bars. However, the same cannot be said about hours, therefore, the tickmarks were explicitly shifted to highlight that the bar in between, say, 14:00 and 15:00 ticks show counts for posts in time range [14:00, 15:00).

Note: by "posting" here we mean the event of a feedback message being visible to the members of channel #15-feedback

In [ ]:
df_datetime = df.dropna(subset=['posting_timestamp']).copy()

df_datetime['DayOfWeek'] = df_datetime['posting_timestamp'].dt.day_name()
labels = ['00:00-03:59', '04:00-07:59', '08:00-11:59', '12:00-15:59', '16:00-19:59', '20:00-23:59']

df_datetime['Hour_4H'] = pd.cut(
    df_datetime['posting_timestamp'].dt.hour, 
    bins=[-1, 3, 7, 11, 15, 19, 23], 
    labels=labels
)

day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

heatmap_data = pd.crosstab(
    df_datetime['DayOfWeek'], 
    df_datetime['Hour_4H']
).reindex(day_order)

plt.figure(figsize=(12, 5))
sns.heatmap(heatmap_data, cmap='YlGnBu', annot=True,  cbar_kws={'label': 'Post Count'})
plt.title('Post Frequency Heatmap (Day of Week vs. Hour of Day)')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

The start of the week has been set to Sunday to maintain the comparability with the previous section.

### Nominal variables

Nominal variables are such categorical variables where categories have no inherent order or ranking.
The univariate analysis here focuses on frequency distributions, central tendency (mode), and cardinality/entropy.

#### survey_type

Just sanity checks, this variable here is not necessary from the point of view of the decision making. We expect it to be not null and constant everywhere; `survey_type='6 Month'`.

In [ ]:
st = df['survey_type']
is_valid = (st.eq('1 Year')).all()
print(f'''
The variable survey_type is valid: {is_valid}
''')

### Ordinal variables

#### organization_rating

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

current_series = df['organization_rating']
total_obs = len(current_series)
ordinal_order = list(current_series.dtype.categories)

freq_map = {}
for item in current_series:
    key = 'Unspecified' if item is None or (hasattr(item, 'isna') and item.isna()) or str(item) in ('nan', 'Unspecified') else str(item)
    freq_map[key] = freq_map.get(key, 0) + 1

categories = [str(cat) for cat in ordinal_order] + ['Unspecified']
counts = [freq_map.get(cat, 0) for cat in categories]

fig, ax = plt.subplots(figsize=(8, 4.5))

palette = ['gray' if cat == 'Unspecified' else 'tab:blue' for cat in categories]

sns.barplot(
    x=categories,
    y=counts,
    hue=categories,
    legend=False,
    ax=ax,
    palette=palette
)

ax.set_title('Organization Rating Frequency Distribution')
ax.set_xlabel('Organization Rating')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)

for container in ax.containers:
    labels = [f'{int(v)}\n({v/total_obs:.2%})' for v in container.datavalues]
    ax.bar_label(container, labels=labels, padding=3, fontsize=8)

ax.set_ylim(0, max(counts, default=1) * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
*ordered_valid, na_counter = list(zip(categories, counts))
na_count = na_counter[1]
average_score = sum(int(cat) * count for (cat, count) in ordered_valid)/total_obs
median_score = current_series.cat.codes[current_series.cat.codes >= 0].median()
mode_key, mode_val = max(ordered_valid, key=lambda x: x[1])
antimode_key, antimode_val = min(ordered_valid, key=lambda x: x[1])


print(f'''
The average score is "{average_score:.2f}".
The median score is "{int(median_score) if int(median_score) == median_score else median_score}".
The mode score is "{mode_key}" with value {mode_val}, constituting therefore {mode_val/total_obs:.2%} of all observations.
The antimode score is "{antimode_key}" with value {antimode_val}, constituting therefore {antimode_val/total_obs:.2%} of all observations.
There are {na_count} answer{'s' if na_count != 1 else ''} with no score specified, constituting therefore {na_count/total_obs:.2%} of all observations.
''')

average_score

### Free text variables

Univariate analysis of free-text (unstructured text) data focuses on summarizing the distribution, statistical properties, length characteristics, and dominant vocabulary or themes of a single text column.

#### positive_feedback

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['positive_feedback']
lang = 'PL'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', '<NA>'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

Lexical Richness (or Lexical Diversity) evaluates the variety and breadth of vocabulary used in a text corpus relative to its total length. Here measured by Corpus and Mean Document-Level TTR.

Type-Token Ratio (TTR):
$$\text{TTR} = \frac{\text{Unique Words (Types)}}{\text{Total Words (Tokens)}}$$

Scale: $0.0$ to $1.0$.

Interpretation: Higher values indicate richer vocabulary. 

However, TTR naturally drops as text length increases because common words repeat.


In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.95, not 0.0.

Note: we need more data to draw conlusions.

#### negative_feedback

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['negative_feedback']
lang = 'PL'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', 'Unspecified'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.85, not 0.0.

Note: we need more data to draw conlusions.

#### positive_feedback_EN

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['positive_feedback_EN']
lang = 'EN'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', '<NA>'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.8, not 0.0.

Note: we need more data to draw conlusions.

#### negative_feedback_EN

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

text_series = df['negative_feedback_EN']
lang = 'EN'

cleaned_docs = []
char_lengths = []
word_counts = []

for doc in text_series:
    if doc is None or (hasattr(doc, 'isna') and doc.isna()) or str(doc) in ('nan', '<NA>'):
        continue
    
    text_str = str(doc).strip()
    if not text_str:
        continue
    
    char_lengths.append(len(text_str))
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]{2,}\b', text_str.lower())
    word_counts.append(len(words))
    cleaned_docs.append(words)

english_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'it', 'this', 'that', 'was'}
polish_stopwords = {
    # Conjunctions & Particles
    'a', 'aby', 'ale', 'ani', 'aż', 'bo', 'bowiem', 'co', 'czy', 'do', 'dla', 'i', 'ich', 
    'ile', 'im', 'iż', 'jak', 'jaka', 'jaki', 'jakie', 'jako', 'jest', 'jeśli', 'jeżeli', 
    'już', 'kazdy', 'każdy', 'kiedy', 'kogo', 'komu', 'który', 'która', 'które', 
    'których', 'którym', 'którzy', 'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 
    'mój', 'moje', 'może', 'mu', 'my', 'na', 'nam', 'nami', 'nas', 'nasi', 
    'nasz', 'nasza', 'nasze', 'natomiast', 'nic', 'nie', 'nim', 'nimi', 'niż', 
    'o', 'od', 'gdy', 'gdyż', 'głównie', 'oraz', 'po', 'pod', 'ponieważ', 'przed', 
    'przeze', 'przy', 'są', 'sam', 'sama', 'sądzę', 'się', 'skąd', 'sobie', 
    'sposób', 'swoje', 'ta', 'tak', 'taka', 'taki', 'takie', 'także', 'tam', 
    'te', 'tego', 'tej', 'temu', 'ten', 'teraz', 'też', 'to', 'tobie', 'tobą', 
    'trzeba', 'tu', 'tutaj', 'twoja', 'twoje', 'twój', 'ty', 'tym', 'tę', 
    'w', 'wam', 'wami', 'was', 'wasz', 'wasza', 'wasze', 'we', 'więc', 'wszyscy', 
    'wszystkie', 'wszystkich', 'wszystkim', 'wszystko', 'wtedy', 'wy', 'z', 
    'za', 'żaden', 'żadna', 'żadne', 'żadnych', 'zaś', 'ze', 'że', 'żeby'
}
stopwords = polish_stopwords if lang == 'PL' else english_stopwords

all_unigrams = [w for doc in cleaned_docs for w in doc if w not in stopwords]
all_bigrams = [f"{doc[i]} {doc[i+1]}" for doc in cleaned_docs for i in range(len(doc)-1) if doc[i] not in stopwords and doc[i+1] not in stopwords]

unigram_counts = Counter(all_unigrams).most_common(10)
bigram_counts = Counter(all_bigrams).most_common(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(word_counts, bins=20, kde=True, ax=axes[0], color='tab:blue')
axes[0].set_title('Word Count Distribution per Document')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for patch in axes[0].patches:
    height = int(patch.get_height())
    if height > 0:
        axes[0].annotate(
            f'{height}',
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 2),
            textcoords='offset points'
        )

axes[0].set_xlim(0, 100)

top_uni_words = [k for k, _ in unigram_counts]
top_uni_vals = [v for _, v in unigram_counts]
sns.barplot(x=top_uni_vals, y=top_uni_words, ax=axes[1], hue=top_uni_words, legend=False, palette='Blues_r')
axes[1].set_title('Top 10 Unigrams (Excl. Stopwords)')
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

top_bi_words = [k for k, _ in bigram_counts]
top_bi_vals = [v for _, v in bigram_counts]
sns.barplot(x=top_bi_vals, y=top_bi_words, ax=axes[2], hue=top_bi_words, legend=False, palette='Blues_r')
axes[2].set_title('Top 10 Bigrams')
axes[2].set_xlabel('Count')
axes[2].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[2].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

for ax in axes[1:]:
    for container in ax.containers:
        labels = [f'{int(v)}' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

all_tokens = [w for doc in cleaned_docs for w in doc]
total_tokens = len(all_tokens)
unique_tokens = len(set(all_tokens))

corpus_ttr = unique_tokens / total_tokens if total_tokens > 0 else 0
doc_ttrs = [len(set(doc)) / len(doc) for doc in cleaned_docs if len(doc) > 0]
mean_doc_ttr = sum(doc_ttrs) / len(doc_ttrs) if doc_ttrs else 0

print(f"""
Total Tokens (Words):      {total_tokens}
Unique Types (Vocab):      {unique_tokens}
Corpus TTR:                {corpus_ttr:.4f}
Mean Document-Level TTR:   {mean_doc_ttr:.4f}
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(doc_ttrs, bins=15, kde=True, ax=ax, color='tab:blue')
ax.set_title('Document-Level Lexical Richness (TTR) Distribution')
ax.set_xlabel('Type-Token Ratio (TTR)')
ax.set_ylabel('Document Count')
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

Note: the x axis starts at 0.75, not 0.0.

Note: we need more data to draw conlusions.

### Continuous variables

When performing univariate analysis on continuous variables, the focus shifts to distribution shape, central tendency, spread, tail behavior, and data integrity.

#### Sentiment variables: [positive|negative]\_feedback_EN\_[neg|neu|pos|compound]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

grid_layout = [
    ['positive_feedback_EN_neg', 'positive_feedback_EN_neu', 'positive_feedback_EN_pos', 'positive_feedback_EN_compound'],
    ['negative_feedback_EN_neg', 'negative_feedback_EN_neu', 'negative_feedback_EN_pos', 'negative_feedback_EN_compound']
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True, sharex='col')

for row_idx in range(2):
    for col_idx in range(4):
        ax = axes[row_idx, col_idx]
        col_name = grid_layout[row_idx][col_idx]
        current_series = df[col_name].dropna()
        
        mean_val = current_series.mean() if not current_series.empty else 0
        median_val = current_series.median() if not current_series.empty else 0
        
        sns.histplot(current_series, kde=True, ax=ax, color='tab:blue', bins=30)
        
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.2, label=f'Mean: {mean_val:.2f}')
        ax.axvline(median_val, color='green', linestyle='-', linewidth=1.2, label=f'Med: {median_val:.2f}')
        
        ax.set_title(col_name, fontsize=9, fontweight='bold')
        ax.set_xlabel('Score' if row_idx == 1 else '')
        ax.set_ylabel('Count' if col_idx == 0 else '')
        ax.legend(loc='upper right', fontsize=8, frameon=True)

for col_idx in range(3):
    axes[0, col_idx].set_xlim(0.0, 1.0)
    axes[1, col_idx].set_xlim(0.0, 1.0)

axes[0, 3].set_xlim(-1.0, 1.0)
axes[1, 3].set_xlim(-1.0, 1.0)

for ax in axes.flatten():
    ax.autoscale(enable=False, axis='x')

plt.tight_layout()
plt.show()

## Bivariate Analysis

## Time series analysis

### How do the sentiment scores evolve over time?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Ensure datetime format
df["posting_timestamp"] = pd.to_datetime(df["posting_timestamp"])
df["date"] = df["posting_timestamp"].dt.date

# 2. Resample daily across the entire company (no department split)
daily_sentiment = (
    df.groupby("date")[
        ["positive_feedback_EN_compound", "negative_feedback_EN_compound"]
    ]
    .mean()
)

# Fill missing calendar days to maintain an accurate 30-day rolling window
full_date_range = pd.date_range(
    start=daily_sentiment.index.min(), end=daily_sentiment.index.max(), freq="D"
)
daily_long = daily_sentiment.reindex(full_date_range).reset_index()

# Fix for reset_index name
daily_long.rename(columns={"level_0": "date", "index": "date"}, inplace=True)

# 3. Calculate 30-day Moving Averages
daily_long["positive_30d_sma"] = (
    daily_long["positive_feedback_EN_compound"]
    .rolling(window=30, min_periods=1)
    .mean()
)
daily_long["negative_30d_sma"] = (
    daily_long["negative_feedback_EN_compound"]
    .rolling(window=30, min_periods=1)
    .mean()
)

# ==========================================
# PLOT 1: Positive Feedback (30-Day Moving Average)
# ==========================================
fig, ax1 = plt.subplots(figsize=(12, 5))

# Raw daily mean (faded)
sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_feedback_EN_compound",
    color="lightgreen",
    alpha=0.35,
    label="Daily Average",
    ax=ax1,
)

# 30-Day SMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_30d_sma",
    color="#1e8449",
    linewidth=2.5,
    label="30-Day Moving Avg",
    ax=ax1,
)

ax1.set_title(
    "Positive Feedback Sentiment: 30-Day Trend", fontsize=13, fontweight="bold"
)
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("VADER Compound Score", fontsize=11)
ax1.set_ylim(-1.05, 1.05)
ax1.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax1.tick_params(axis="x", rotation=30)
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()


# ==========================================
# PLOT 2: Negative Feedback (30-Day Moving Average)
# ==========================================
fig, ax2 = plt.subplots(figsize=(12, 5))

# Raw daily mean (faded)
sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_feedback_EN_compound",
    color="salmon",
    alpha=0.35,
    label="Daily Average",
    ax=ax2,
)

# 30-Day SMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_30d_sma",
    color="#78281f",
    linewidth=2.5,
    label="30-Day Moving Avg",
    ax=ax2,
)

ax2.set_title(
    "Negative Feedback Sentiment: 30-Day Trend", fontsize=13, fontweight="bold"
)
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("VADER Compound Score", fontsize=11)
ax2.set_ylim(-1.05, 1.05)
ax2.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax2.tick_params(axis="x", rotation=30)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()

Due to low answers volume and short overall time period, the simple moving average is chaotic for both feedback sentiment scores.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Prepare base DataFrame and Datetime columns
df_clean = df.dropna(subset=["posting_timestamp"]).copy()
df_clean["posting_timestamp"] = pd.to_datetime(df_clean["posting_timestamp"])
df_clean["date"] = df_clean["posting_timestamp"].dt.date
df_clean["Year-Month"] = df_clean["posting_timestamp"].dt.strftime("%Y-%m")

# 2. Compute Monthly Post Counts
monthly_counts = df_clean.groupby("Year-Month").size().to_dict()

# 3. Resample daily across full date range
daily_sentiment = (
    df_clean.groupby("date")[
        ["positive_feedback_EN_compound", "negative_feedback_EN_compound"]
    ]
    .mean()
)

full_date_range = pd.date_range(
    start=daily_sentiment.index.min(), end=daily_sentiment.index.max(), freq="D"
)
daily_long = daily_sentiment.reindex(full_date_range).reset_index()
daily_long.rename(columns={"level_0": "date", "index": "date"}, inplace=True)

daily_long["Year-Month"] = daily_long["date"].dt.strftime("%Y-%m")
daily_long["monthly_post_count"] = daily_long["Year-Month"].map(monthly_counts).fillna(0)

# 4. Calculate 30-day Exponential Moving Averages (EMA)
daily_long["positive_30d_ema"] = (
    daily_long["positive_feedback_EN_compound"]
    .ewm(span=30, adjust=False)
    .mean()
)
daily_long["negative_30d_ema"] = (
    daily_long["negative_feedback_EN_compound"]
    .ewm(span=60, adjust=False)
    .mean()
)


# ==========================================
# PLOT 1: Positive Feedback (30-Day EMA) + Monthly Volume
# ==========================================
fig, ax1 = plt.subplots(figsize=(13, 6))

# Secondary Y-Axis for Volume
ax1_twin = ax1.twinx()
ax1_twin.grid(False)
ax1_twin.bar(
    daily_long["date"],
    daily_long["monthly_post_count"],
    color="#3498db",
    alpha=0.18,
    width=1.0,
    label="Monthly Post Volume",
)
ax1_twin.set_ylabel("Monthly Post Count", color="#2980b9", fontsize=11)
ax1_twin.tick_params(axis="y", labelcolor="#2980b9")

# Primary Y-Axis: Raw Daily vs. 30-Day EMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_feedback_EN_compound",
    color="lightgreen",
    alpha=0.35,
    label="Daily Avg Score",
    ax=ax1,
)

sns.lineplot(
    data=daily_long,
    x="date",
    y="positive_30d_ema",
    color="#1e8449",
    linewidth=2.5,
    label="30-Day Exponential Moving Avg (EMA)",
    ax=ax1,
)

ax1.set_title(
    "Positive Feedback Sentiment (30-Day EMA) & Monthly Volume",
    fontsize=13,
    fontweight="bold",
)
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("VADER Compound Score", fontsize=11)
ax1.set_ylim(-1.05, 1.05)
ax1.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax1.tick_params(axis="x", rotation=30)
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()


# ==========================================
# PLOT 2: Negative Feedback (30-Day EMA) + Monthly Volume
# ==========================================
fig, ax2 = plt.subplots(figsize=(13, 6))

# Secondary Y-Axis for Volume
ax2_twin = ax2.twinx()
ax2_twin.grid(False)
ax2_twin.bar(
    daily_long["date"],
    daily_long["monthly_post_count"],
    color="#3498db",
    alpha=0.18,
    width=1.0,
    label="Monthly Post Volume",
)
ax2_twin.set_ylabel("Monthly Post Count", color="#2980b9", fontsize=11)
ax2_twin.tick_params(axis="y", labelcolor="#2980b9")

# Primary Y-Axis: Raw Daily vs. 30-Day EMA
sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_feedback_EN_compound",
    color="salmon",
    alpha=0.35,
    label="Daily Avg Score",
    ax=ax2,
)

sns.lineplot(
    data=daily_long,
    x="date",
    y="negative_30d_ema",
    color="#78281f",
    linewidth=2.5,
    label="30-Day Exponential Moving Avg (EMA)",
    ax=ax2,
)

ax2.set_title(
    "Negative Feedback Sentiment (30-Day EMA) & Monthly Volume",
    fontsize=13,
    fontweight="bold",
)
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("VADER Compound Score", fontsize=11)
ax2.set_ylim(-1.05, 1.05)
ax2.axhline(0, color="gray", linestyle=":", alpha=0.6)
ax2.tick_params(axis="x", rotation=30)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()

Note: we need more data to draw conlusions.

### How does the organization rating evolve over time?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

df["organization_rating"] = df["organization_rating"].astype(float)
df_clean = df.dropna(subset=["posting_timestamp"]).copy()
df_clean["posting_timestamp"] = pd.to_datetime(df_clean["posting_timestamp"])
df_clean["date"] = df_clean["posting_timestamp"].dt.date
df_clean["Year-Month"] = df_clean["posting_timestamp"].dt.strftime("%Y-%m")

monthly_counts = df_clean.groupby("Year-Month").size().to_dict()

daily_rating = df_clean.groupby("date")["organization_rating"].mean()

full_date_range = pd.date_range(
    start=daily_rating.index.min(), end=daily_rating.index.max(), freq="D"
)
daily_df = daily_rating.reindex(full_date_range).reset_index()
daily_df.rename(columns={"level_0": "date", "index": "date"}, inplace=True)

daily_df["Year-Month"] = daily_df["date"].dt.strftime("%Y-%m")
daily_df["monthly_post_count"] = daily_df["Year-Month"].map(monthly_counts).fillna(0)

daily_df["rating_30d_ema"] = (
    daily_df["organization_rating"]
    .ewm(span=30, adjust=False)
    .mean()
)

fig, ax1 = plt.subplots(figsize=(13, 6))

ax2 = ax1.twinx()
ax2.grid(False)
ax2.bar(
    daily_df["date"],
    daily_df["monthly_post_count"],
    color="#3498db",
    alpha=0.18,
    width=1.0,
    label="Monthly Post Volume",
)
ax2.set_ylabel("Monthly Post Count", color="#2980b9", fontsize=11)
ax2.tick_params(axis="y", labelcolor="#2980b9")

sns.lineplot(
    data=daily_df,
    x="date",
    y="organization_rating",
    color="skyblue",
    alpha=0.4,
    label="Daily Avg Rating",
    ax=ax1,
)

sns.lineplot(
    data=daily_df,
    x="date",
    y="rating_30d_ema",
    color="#2980b9",
    linewidth=2.5,
    label="30-Day EMA",
    ax=ax1,
)

ax1.set_title(
    "Organization Rating (1-5 Scale): 30-Day EMA & Monthly Volume",
    fontsize=13,
    fontweight="bold",
)
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("Rating Scale (1 - 5)", fontsize=11)
ax1.set_ylim(1.0, 5.0)
ax1.axhline(3.0, color="gray", linestyle=":", alpha=0.6)
ax1.tick_params(axis="x", rotation=30)
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()

Note: we need more data to draw conlusions.